# 第11回: Diffusion Policy - 拡散モデルによる行動生成

このノートブックでは、ロボット制御のための **Diffusion Policy** を学びます。
画像生成で大きな成功を収めた拡散モデル（Diffusion Models）を、ロボットの行動生成に応用する手法です。

**学習内容:**
1. DDPM（Denoising Diffusion Probabilistic Models）の仕組み
2. DDIM（Denoising Diffusion Implicit Models）による高速推論
3. 1D U-Net によるノイズ予測ネットワーク
4. Diffusion Policy の学習と推論
5. EMA（指数移動平均）による学習安定化
6. 予測結果の可視化と推論時間の比較

## 0. 環境準備

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

---
## 1. 拡散モデル（Diffusion Models）の基礎

### 1.1 DDPM（Denoising Diffusion Probabilistic Models）

DDPMは2つの過程から構成されます。

#### 拡散過程（Forward Process）
データ $x_0$ にガウスノイズを段階的に加えていきます:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I)$$

ここで $\beta_t$ はノイズスケジュールと呼ばれ、各ステップで加えるノイズの大きさを制御します。
$\bar{\alpha}_t = \prod_{s=1}^{t} (1-\beta_s)$ と定義すると、任意のステップ $t$ のノイズ付きデータを一度に計算できます:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t) I)$$

つまり: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$

#### 逆拡散過程（Reverse Process）
ニューラルネットワークでノイズを段階的に除去します:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t),\; \sigma_t^2 I)$$

#### 学習目標
ネットワーク $\epsilon_\theta$ は、加えられたノイズを予測するように学習します:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

### 1.2 DDIM（Denoising Diffusion Implicit Models）

DDIMはDDPMと同じ学習済みモデルを使いつつ、推論を高速化する手法です。

**DDPMとの主な違い:**
- **非マルコフ的な逆過程**: DDPMがマルコフ連鎖に基づくのに対し、DDIMは非マルコフ的な逆過程を定義
- **少ないステップ数で推論可能**: DDPMが全 $T$ ステップ必要なのに対し、DDIMは $T$ のサブセット（例: 10ステップ）で推論可能
- **決定論的サンプリング**: ノイズ項を除くことで、同じ初期ノイズから常に同じ結果を生成可能

DDIMの更新式:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{predicted } x_0} + \sqrt{1-\bar{\alpha}_{t-1}}\, \epsilon_\theta(x_t, t)$$

### 1.3 ノイズスケジュールの可視化

In [ ]:
T = 100
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(betas.numpy())
axes[0].set_title("ノイズスケジュール β_t")
axes[0].set_xlabel("ステップ t")
axes[0].set_ylabel("β_t")

axes[1].plot(alphas_cumprod.numpy())
axes[1].set_title("累積積 ᾱ_t")
axes[1].set_xlabel("ステップ t")
axes[1].set_ylabel("ᾱ_t")

axes[2].plot(alphas_cumprod.sqrt().numpy(), label="√ᾱ_t (信号)")
axes[2].plot((1 - alphas_cumprod).sqrt().numpy(), label="√(1-ᾱ_t) (ノイズ)")
axes[2].set_title("信号とノイズの比率")
axes[2].set_xlabel("ステップ t")
axes[2].legend()

plt.tight_layout()
plt.show()

### 1.4 拡散過程の可視化（1次元信号）

In [ ]:
# 1次元の行動系列（サイン波）をサンプルデータとする
horizon = 16
action_dim = 2
x0 = torch.zeros(1, horizon, action_dim)
x0[0, :, 0] = torch.sin(torch.linspace(0, 2 * math.pi, horizon))
x0[0, :, 1] = torch.cos(torch.linspace(0, 2 * math.pi, horizon))

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
show_steps = [0, 10, 25, 50, 99]

for i, t_val in enumerate(show_steps):
    sqrt_ab = alphas_cumprod[t_val].sqrt()
    sqrt_1_ab = (1 - alphas_cumprod[t_val]).sqrt()
    noise = torch.randn_like(x0)
    xt = sqrt_ab * x0 + sqrt_1_ab * noise

    for dim_idx in range(2):
        axes[dim_idx, i].plot(x0[0, :, dim_idx].numpy(), "b--", alpha=0.5, label="元の信号")
        axes[dim_idx, i].plot(xt[0, :, dim_idx].numpy(), "r-", label="ノイズ付き")
        axes[dim_idx, i].set_title(f"t={t_val}, dim={dim_idx}")
        axes[dim_idx, i].set_ylim(-3, 3)
        if i == 0:
            axes[dim_idx, i].legend(fontsize=8)

plt.suptitle("拡散過程: 行動系列へのノイズ付加", fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. ノイズ予測ネットワーク（1D U-Net）

Diffusion Policyでは、ノイズ予測ネットワークとして **1D U-Net** を使用します。
U-Netは画像セグメンテーションで広く使われるアーキテクチャで、
エンコーダ・デコーダ構造にスキップ接続を加えたものです。

ここでは以下の3つのコンポーネントを実装します:
1. **SinusoidalPositionEmbedding**: 拡散ステップ $t$ をベクトルに変換
2. **ConditionalResBlock1D**: 条件付き残差ブロック（観測と時刻で条件付け）
3. **NoisePredictor**: 1D U-Net 全体

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """拡散ステップ t を正弦波位置埋め込みに変換する。

    Transformerの位置エンコーディングと同じ考え方で、
    整数のステップ番号を連続的なベクトル表現に変換します。
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

# 動作確認
time_embed = SinusoidalPositionEmbedding(128)
t_test = torch.tensor([0.0, 25.0, 50.0, 99.0])
emb_out = time_embed(t_test)
print(f"入力: t = {t_test.tolist()}")
print(f"出力形状: {emb_out.shape}")  # (4, 128)

In [ ]:
# 位置埋め込みの可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 各ステップの埋め込みベクトル
t_range = torch.arange(0, 100, dtype=torch.float)
embeddings = time_embed(t_range).detach().numpy()

axes[0].imshow(embeddings.T, aspect="auto", cmap="coolwarm")
axes[0].set_xlabel("ステップ t")
axes[0].set_ylabel("埋め込み次元")
axes[0].set_title("時刻埋め込み（全ステップ）")

# 選択したステップの埋め込みを比較
for t_val in [0, 25, 50, 75, 99]:
    axes[1].plot(embeddings[t_val, :32], label=f"t={t_val}")
axes[1].set_xlabel("次元（最初の32次元）")
axes[1].set_ylabel("値")
axes[1].set_title("各ステップの埋め込みベクトル")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
class ConditionalResBlock1D(nn.Module):
    """条件付き1D残差ブロック。

    観測情報と拡散ステップの情報で条件付けされた残差ブロックです。
    条件情報は中間特徴量に加算されます。
    """
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.Mish()
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.Mish()
        )
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        h = self.block1(x)
        # 条件情報を特徴量に加算（FiLMスタイルの条件付け）
        h = h + self.cond_proj(cond).unsqueeze(-1)
        h = self.block2(h)
        return h + self.residual(x)

In [ ]:
class NoisePredictor(nn.Module):
    """1D U-Net によるノイズ予測ネットワーク。

    入力:
        x: ノイズ付き行動系列 (B, horizon, action_dim)
        t: 拡散ステップ (B,)
        obs: 観測ベクトル (B, obs_dim)
    出力:
        predicted_noise: 予測ノイズ (B, horizon, action_dim)
    """
    def __init__(self, action_dim, obs_dim, horizon, diffusion_step_embed_dim=128):
        super().__init__()
        cond_dim = diffusion_step_embed_dim + obs_dim
        self.time_embed = SinusoidalPositionEmbedding(diffusion_step_embed_dim)
        self.obs_proj = nn.Linear(obs_dim, obs_dim)

        # Down (エンコーダ)
        self.down1 = ConditionalResBlock1D(action_dim, 64, cond_dim)
        self.down2 = ConditionalResBlock1D(64, 128, cond_dim)
        # Mid (ボトルネック)
        self.mid = ConditionalResBlock1D(128, 128, cond_dim)
        # Up (デコーダ) - スキップ接続で入力チャネル数が倍
        self.up2 = ConditionalResBlock1D(256, 64, cond_dim)   # 128+128 skip
        self.up1 = ConditionalResBlock1D(128, 64, cond_dim)   # 64+64 skip
        self.final = nn.Conv1d(64, action_dim, 1)

    def forward(self, x, t, obs):
        # 条件情報の構築
        t_emb = self.time_embed(t)
        obs_emb = self.obs_proj(obs)
        cond = torch.cat([t_emb, obs_emb], dim=-1)

        # (B, horizon, action_dim) -> (B, action_dim, horizon)
        x = x.transpose(1, 2)

        # エンコーダ
        h1 = self.down1(x, cond)
        h2 = self.down2(h1, cond)

        # ボトルネック
        h = self.mid(h2, cond)

        # デコーダ（スキップ接続）
        h = self.up2(torch.cat([h, h2], dim=1), cond)
        h = self.up1(torch.cat([h, h1], dim=1), cond)

        # (B, action_dim, horizon) -> (B, horizon, action_dim)
        return self.final(h).transpose(1, 2)

# 動作確認
action_dim = 2
obs_dim = 10
horizon = 16
model = NoisePredictor(action_dim, obs_dim, horizon).to(device)

x_test = torch.randn(4, horizon, action_dim).to(device)
t_test = torch.tensor([0.0, 25.0, 50.0, 99.0]).to(device)
obs_test = torch.randn(4, obs_dim).to(device)

out = model(x_test, t_test, obs_test)
print(f"入力形状: x={x_test.shape}, t={t_test.shape}, obs={obs_test.shape}")
print(f"出力形状: {out.shape}")  # (4, 16, 2)
total_params = sum(p.numel() for p in model.parameters())
print(f"パラメータ数: {total_params:,}")

---
## 3. Diffusion Policy（DDPMとDDIMの統合）

Diffusion Policyは、拡散モデルをロボットの行動生成に応用したものです。
観測 $o$ を条件として、行動系列 $a_{1:H}$ を生成します。

- **学習時**: 行動系列にノイズを加え、ネットワークにそのノイズを予測させる
- **推論時**: ランダムノイズから出発し、逆拡散過程で行動系列を生成

DDPMとDDIMの両方のサンプリング方法を実装します。

In [ ]:
class DiffusionPolicy:
    """Diffusion Policyの実装（DDPMとDDIMをサポート）。"""

    def __init__(self, model, num_diffusion_steps=100, beta_start=1e-4, beta_end=0.02):
        self.model = model
        self.T = num_diffusion_steps
        self.device = next(model.parameters()).device

        # DDPMノイズスケジュール
        self.betas = torch.linspace(beta_start, beta_end, num_diffusion_steps).to(self.device)
        self.alphas = (1 - self.betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)

    def add_noise(self, x0, t, noise=None):
        """拡散過程: x0 にステップ t に対応するノイズを付加する。"""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = self.alphas_cumprod[t].sqrt()
        sqrt_one_minus = (1 - self.alphas_cumprod[t]).sqrt()
        # ブロードキャストのため次元を追加
        return sqrt_alpha_bar[:, None, None] * x0 + sqrt_one_minus[:, None, None] * noise, noise

    def train_step(self, actions, obs, optimizer):
        """1回の学習ステップ。"""
        # ランダムな拡散ステップを選択
        t = torch.randint(0, self.T, (actions.shape[0],), device=actions.device)
        # ノイズを付加
        noisy_actions, noise = self.add_noise(actions, t)
        # ノイズを予測
        predicted_noise = self.model(noisy_actions, t.float(), obs)
        # MSE損失
        loss = F.mse_loss(predicted_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        return loss.item()

    @torch.no_grad()
    def sample_ddpm(self, obs, shape):
        """DDPM サンプリング: 全ステップを逆向きに辿る。"""
        x = torch.randn(shape, device=self.device)
        for t in reversed(range(self.T)):
            t_batch = torch.full((shape[0],), t, device=self.device, dtype=torch.float)
            noise_pred = self.model(x, t_batch, obs)

            alpha = self.alphas[t]
            alpha_bar = self.alphas_cumprod[t]
            # DDPM更新式
            x = (1 / alpha.sqrt()) * (x - (1 - alpha) / (1 - alpha_bar).sqrt() * noise_pred)
            if t > 0:
                x += self.betas[t].sqrt() * torch.randn_like(x)
        return x

    @torch.no_grad()
    def sample_ddim(self, obs, shape, num_steps=10):
        """DDIM サンプリング: サブサンプルされたステップで高速推論。"""
        step_size = self.T // num_steps
        timesteps = list(range(0, self.T, step_size))[::-1]

        x = torch.randn(shape, device=self.device)
        for i, t in enumerate(timesteps):
            t_batch = torch.full((shape[0],), t, device=self.device, dtype=torch.float)
            noise_pred = self.model(x, t_batch, obs)

            alpha_bar_t = self.alphas_cumprod[t]
            alpha_bar_prev = self.alphas_cumprod[timesteps[i+1]] if i + 1 < len(timesteps) else torch.tensor(1.0).to(self.device)

            # x0 の予測
            x0_pred = (x - (1 - alpha_bar_t).sqrt() * noise_pred) / alpha_bar_t.sqrt()
            # DDIM更新式（決定論的）
            x = alpha_bar_prev.sqrt() * x0_pred + (1 - alpha_bar_prev).sqrt() * noise_pred
        return x

---
## 4. 学習デモ

簡単なデータセットでDiffusion Policyを学習させてみましょう。
観測ベクトルに基づいて、対応する行動系列（サイン波）を生成するタスクです。

In [ ]:
# データセットの生成
def generate_demo_data(n_samples=1000, obs_dim=10, action_dim=2, horizon=16):
    """デモ用データ: 観測に応じた行動系列を生成。"""
    obs = torch.randn(n_samples, obs_dim)
    actions = torch.zeros(n_samples, horizon, action_dim)

    # 観測の最初の要素で振幅と周波数を決定
    amplitude = obs[:, 0:1].abs().unsqueeze(1)  # (N, 1, 1)
    frequency = 1 + obs[:, 1:2].abs().unsqueeze(1)  # (N, 1, 1)
    t_axis = torch.linspace(0, 2 * math.pi, horizon).unsqueeze(0).unsqueeze(-1)

    actions[:, :, 0:1] = amplitude * torch.sin(frequency * t_axis)
    actions[:, :, 1:2] = amplitude * torch.cos(frequency * t_axis)

    return obs, actions

obs_data, action_data = generate_demo_data()
print(f"観測データ: {obs_data.shape}")
print(f"行動データ: {action_data.shape}")

In [ ]:
# モデルと Diffusion Policy の初期化
action_dim = 2
obs_dim = 10
horizon = 16

model = NoisePredictor(action_dim, obs_dim, horizon).to(device)
policy = DiffusionPolicy(model, num_diffusion_steps=100)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 学習ループ
obs_data_d = obs_data.to(device)
action_data_d = action_data.to(device)
n_epochs = 100
batch_size = 128
losses = []

for epoch in range(n_epochs):
    epoch_losses = []
    indices = torch.randperm(len(obs_data_d))
    for start in range(0, len(obs_data_d), batch_size):
        idx = indices[start:start + batch_size]
        loss = policy.train_step(action_data_d[idx], obs_data_d[idx], optimizer)
        epoch_losses.append(loss)
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("学習曲線")
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. EMA（Exponential Moving Average）

EMA はモデルパラメータの指数移動平均を保持するテクニックです。

$$\theta_{\text{EMA}}^{(t)} = \gamma \cdot \theta_{\text{EMA}}^{(t-1)} + (1-\gamma) \cdot \theta^{(t)}$$

ここで $\gamma$（通常 0.995 ~ 0.9999）は減衰率です。

**EMAの利点:**
- 学習中のパラメータの振動を平滑化
- 推論時にEMAパラメータを使うことで、より安定した予測が得られる
- 汎化性能の向上が期待できる

In [ ]:
class EMA:
    """Exponential Moving Average（指数移動平均）。

    学習中にモデルパラメータの移動平均を保持し、
    推論時にはその平均パラメータを使用します。
    """
    def __init__(self, model, decay=0.995):
        self.model = model
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            self.shadow[name] = param.data.clone()

    def update(self):
        """学習ステップ毎に呼び出し、移動平均を更新する。"""
        for name, param in self.model.named_parameters():
            self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data

    def apply(self):
        """推論時にEMAパラメータを適用する。元のパラメータはバックアップされる。"""
        self.backup = {}
        for name, param in self.model.named_parameters():
            self.backup[name] = param.data.clone()
            param.data = self.shadow[name]

    def restore(self):
        """EMA適用後に元のパラメータに戻す。"""
        for name, param in self.model.named_parameters():
            param.data = self.backup[name]

### EMAを使った学習

In [ ]:
# EMA付きで再学習
model_ema = NoisePredictor(action_dim, obs_dim, horizon).to(device)
policy_ema = DiffusionPolicy(model_ema, num_diffusion_steps=100)
optimizer_ema = torch.optim.Adam(model_ema.parameters(), lr=1e-3)
ema = EMA(model_ema, decay=0.995)

losses_ema = []

for epoch in range(n_epochs):
    epoch_losses = []
    indices = torch.randperm(len(obs_data_d))
    for start in range(0, len(obs_data_d), batch_size):
        idx = indices[start:start + batch_size]
        loss = policy_ema.train_step(action_data_d[idx], obs_data_d[idx], optimizer_ema)
        epoch_losses.append(loss)
        ema.update()  # 各ステップでEMAを更新
    avg_loss = np.mean(epoch_losses)
    losses_ema.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(losses, label="EMAなし")
plt.plot(losses_ema, label="EMAあり")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("学習曲線の比較")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 6. 予測結果の可視化

学習済みモデルを使って、DDPMとDDIMの両方で行動系列を生成し比較します。

In [ ]:
# テストデータの準備
test_idx = 0
test_obs = obs_data_d[test_idx:test_idx+1]
true_action = action_data_d[test_idx:test_idx+1]
shape = (1, horizon, action_dim)

# 通常モデルで予測
model.eval()
pred_ddpm = policy.sample_ddpm(test_obs, shape)
pred_ddim = policy.sample_ddim(test_obs, shape, num_steps=10)

# EMAモデルで予測
ema.apply()
pred_ddpm_ema = policy_ema.sample_ddpm(test_obs, shape)
pred_ddim_ema = policy_ema.sample_ddim(test_obs, shape, num_steps=10)
ema.restore()

# 可視化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
titles = ["DDPM（EMAなし）", "DDIM（EMAなし）", "DDPM（EMAあり）", "DDIM（EMAあり）"]
preds = [pred_ddpm, pred_ddim, pred_ddpm_ema, pred_ddim_ema]

for idx, (ax, title, pred) in enumerate(zip(axes.flat, titles, preds)):
    pred_np = pred[0].cpu().numpy()
    true_np = true_action[0].cpu().numpy()

    for d in range(action_dim):
        ax.plot(true_np[:, d], "--", alpha=0.7, label=f"真値 dim{d}")
        ax.plot(pred_np[:, d], "-", alpha=0.9, label=f"予測 dim{d}")
    ax.set_title(title)
    ax.set_xlabel("ステップ")
    ax.set_ylabel("行動値")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("DDPMとDDIMの予測比較", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 複数サンプルの生成（確率的な行動生成の可視化）
n_gen = 8
shape_multi = (n_gen, horizon, action_dim)

test_obs_multi = test_obs.repeat(n_gen, 1)
samples_ddpm = policy.sample_ddpm(test_obs_multi, shape_multi)
samples_ddim = policy.sample_ddim(test_obs_multi, shape_multi, num_steps=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
true_np = true_action[0].cpu().numpy()

for ax, samples, title in zip(axes, [samples_ddpm, samples_ddim], ["DDPM", "DDIM"]):
    for i in range(n_gen):
        s = samples[i].cpu().numpy()
        ax.plot(s[:, 0], alpha=0.4, color="blue")
        ax.plot(s[:, 1], alpha=0.4, color="orange")
    ax.plot(true_np[:, 0], "b--", linewidth=2, label="真値 dim0")
    ax.plot(true_np[:, 1], "r--", linewidth=2, label="真値 dim1")
    ax.set_title(f"{title}: 複数サンプルの重ね描き")
    ax.set_xlabel("ステップ")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. DDPMとDDIMの推論時間比較

DDIMの最大の利点は推論の高速化です。
ステップ数を変えて推論時間と生成品質を比較します。

In [ ]:
# 推論時間のベンチマーク
model.eval()
test_obs_bench = obs_data_d[:16]
shape_bench = (16, horizon, action_dim)
test_obs_bench_expanded = test_obs_bench

# DDPM（全100ステップ）
start = time.time()
for _ in range(3):
    _ = policy.sample_ddpm(test_obs_bench_expanded, shape_bench)
ddpm_time = (time.time() - start) / 3

# DDIM（各ステップ数）
ddim_steps_list = [5, 10, 20, 50]
ddim_times = []
ddim_errors = []

for n_steps in ddim_steps_list:
    start = time.time()
    for _ in range(3):
        pred = policy.sample_ddim(test_obs_bench_expanded, shape_bench, num_steps=n_steps)
    elapsed = (time.time() - start) / 3
    ddim_times.append(elapsed)

    # 予測品質（MSE）
    error = F.mse_loss(pred, action_data_d[:16]).item()
    ddim_errors.append(error)

# 結果表示
print(f"{'方法':<20} {'時間 (ms)':<15} {'MSE':<10}")
print("-" * 45)
print(f"{'DDPM (100 steps)':<20} {ddpm_time*1000:<15.1f} {'---':<10}")
for n_steps, t_val, err in zip(ddim_steps_list, ddim_times, ddim_errors):
    print(f"{'DDIM (' + str(n_steps) + ' steps)':<20} {t_val*1000:<15.1f} {err:<10.6f}")

In [ ]:
# 推論時間とステップ数のプロット
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 推論時間
axes[0].bar(["DDPM\n(100)"] + [f"DDIM\n({s})" for s in ddim_steps_list],
           [ddpm_time * 1000] + [t * 1000 for t in ddim_times],
           color=["#e74c3c"] + ["#3498db"] * len(ddim_steps_list))
axes[0].set_ylabel("推論時間 (ms)")
axes[0].set_title("推論時間の比較")
axes[0].grid(True, alpha=0.3, axis="y")

# ステップ数 vs MSE
axes[1].plot(ddim_steps_list, ddim_errors, "o-", color="#3498db", linewidth=2)
axes[1].set_xlabel("DDIMステップ数")
axes[1].set_ylabel("MSE")
axes[1].set_title("DDIMステップ数と予測品質のトレードオフ")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. 演習問題

### 演習1: DDPMの `add_noise` と `sample_ddpm` を実装せよ

以下のクラスの `add_noise` メソッドと `sample_ddpm` メソッドを完成させてください。

**ヒント:**
- `add_noise`: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$
- `sample_ddpm`: $x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right) + \sigma_t z$

上記の数式に従って、`add_noise` と `sample_ddpm` を実装してください。

In [ ]:
class MyDiffusionPolicy:
    def __init__(self, model, num_diffusion_steps=100, beta_start=1e-4, beta_end=0.02):
        self.model = model
        self.T = num_diffusion_steps
        self.device = next(model.parameters()).device
        self.betas = torch.linspace(beta_start, beta_end, num_diffusion_steps).to(self.device)
        self.alphas = (1 - self.betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)

    def add_noise(self, x0, t, noise=None):
        """x0 にステップ t に対応するノイズを付加する。"""
        # ここにコードを書いてください
        pass

    @torch.no_grad()
    def sample_ddpm(self, obs, shape):
        """DDPMサンプリング。"""
        # ここにコードを書いてください
        pass

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyDiffusionPolicy:
    def __init__(self, model, num_diffusion_steps=100, beta_start=1e-4, beta_end=0.02):
        self.model = model
        self.T = num_diffusion_steps
        self.device = next(model.parameters()).device
        self.betas = torch.linspace(beta_start, beta_end, num_diffusion_steps).to(self.device)
        self.alphas = (1 - self.betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = self.alphas_cumprod[t].sqrt()
        sqrt_one_minus = (1 - self.alphas_cumprod[t]).sqrt()
        return sqrt_alpha_bar[:, None, None] * x0 + sqrt_one_minus[:, None, None] * noise, noise

    @torch.no_grad()
    def sample_ddpm(self, obs, shape):
        x = torch.randn(shape, device=self.device)
        for t in reversed(range(self.T)):
            t_batch = torch.full((shape[0],), t, device=self.device, dtype=torch.float)
            noise_pred = self.model(x, t_batch, obs)
            alpha = self.alphas[t]
            alpha_bar = self.alphas_cumprod[t]
            x = (1 / alpha.sqrt()) * (x - (1 - alpha) / (1 - alpha_bar).sqrt() * noise_pred)
            if t > 0:
                x += self.betas[t].sqrt() * torch.randn_like(x)
        return x

```

</details>

### 演習2: EMAクラスを実装し、学習中にEMAを適用せよ

EMA（指数移動平均）クラスの `update`、`apply`、`restore` メソッドを実装してください。
そして学習ループ中にEMAを適用し、EMAの有無で予測品質を比較してください。

**ヒント:**
- `update`: $\theta_{\text{shadow}} \leftarrow \gamma \cdot \theta_{\text{shadow}} + (1-\gamma) \cdot \theta$
- `apply`: 現在のパラメータをバックアップし、shadow パラメータに差し替える
- `restore`: バックアップからパラメータを復元する

EMAクラスの3つのメソッドを実装してください。

In [ ]:
class MyEMA:
    def __init__(self, model, decay=0.995):
        self.model = model
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            self.shadow[name] = param.data.clone()

    def update(self):
        # ここにコードを書いてください
        pass

    def apply(self):
        # ここにコードを書いてください
        pass

    def restore(self):
        # ここにコードを書いてください
        pass

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyEMA:
    def __init__(self, model, decay=0.995):
        self.model = model
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data

    def apply(self):
        self.backup = {}
        for name, param in self.model.named_parameters():
            self.backup[name] = param.data.clone()
            param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            param.data = self.backup[name]

```

</details>

### 演習3: DDPMとDDIMの推論時間を比較し、ステップ数と品質のトレードオフを調べよ

DDIMのステップ数を `[2, 5, 10, 20, 50, 100]` と変化させて、
推論時間と予測品質（MSE）の関係を調べてください。
また、DDPMの結果と比較してください。

DDIMの各ステップ数について推論時間とMSEを計測し、グラフにプロットしてください。

In [ ]:
ddim_steps_range = [2, 5, 10, 20, 50, 100]
# ここにコードを書いてください
# 各ステップ数で推論時間とMSEを計測し、結果をプロットしてください

<details>
<summary><b>回答例を表示</b></summary>

```python
ddim_steps_range = [2, 5, 10, 20, 50, 100]
times_list = []
errors_list = []

model.eval()
test_obs_ex = obs_data_d[:32]
shape_ex = (32, horizon, action_dim)

for n_steps in ddim_steps_range:
    start = time.time()
    for _ in range(5):
        pred = policy.sample_ddim(test_obs_ex, shape_ex, num_steps=n_steps)
    elapsed = (time.time() - start) / 5
    times_list.append(elapsed)
    errors_list.append(F.mse_loss(pred, action_data_d[:32]).item())

# DDPM
start = time.time()
for _ in range(5):
    pred_ddpm_ex = policy.sample_ddpm(test_obs_ex, shape_ex)
ddpm_time_ex = (time.time() - start) / 5

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ddim_steps_range, [t * 1000 for t in times_list], "o-", label="DDIM")
axes[0].axhline(ddpm_time_ex * 1000, color="red", linestyle="--", label="DDPM (100 steps)")
axes[0].set_xlabel("DDIMステップ数")
axes[0].set_ylabel("推論時間 (ms)")
axes[0].set_title("ステップ数 vs 推論時間")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ddim_steps_range, errors_list, "o-", label="DDIM")
axes[1].set_xlabel("DDIMステップ数")
axes[1].set_ylabel("MSE")
axes[1].set_title("ステップ数 vs 予測品質")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

```

</details>

---
## まとめ

このノートブックでは、以下の内容を学びました:

1. **DDPM（Denoising Diffusion Probabilistic Models）**: ガウスノイズの段階的な付加と除去による生成モデル
2. **DDIM（Denoising Diffusion Implicit Models）**: DDPMの高速推論版、少ないステップで推論可能
3. **1D U-Net**: 行動系列のノイズ予測ネットワーク（スキップ接続付きエンコーダ・デコーダ）
4. **Diffusion Policy**: 拡散モデルをロボットの行動生成に応用
5. **EMA**: パラメータの指数移動平均による学習安定化
6. **DDPMとDDIMのトレードオフ**: 推論時間と品質のバランス

**Diffusion Policyの利点:**
- マルチモーダルな行動分布を自然に表現できる
- 学習が安定（モード崩壊が起きにくい）
- 高次元の行動系列を直接生成可能

**注意点:**
- 推論時間がBCやVAEに比べて長い（DDIMで改善可能）
- ノイズスケジュールやステップ数のチューニングが必要